In [ ]:
# 1. Applying Weighted Ensemble on the 2 best performing ML model after applying balancing
# Finding the best weight for each model

# Weighted
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import ADASYN

df = pd.read_csv('heart.csv')
le = LabelEncoder()
for col in ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']:
    df[col] = le.fit_transform(df[col])
X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=369)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- TRAIN MODELS ON THEIR SPECIFIC DATA ---
# 1. LightGBM (SMOTETomek)
smt = SMOTETomek(random_state=369)
X_train_lgbm, y_train_lgbm = smt.fit_resample(X_train_scaled, y_train)
model_lgbm = LGBMClassifier(random_state=369, verbose=-1).fit(X_train_lgbm, y_train_lgbm)

# 2. XGBoost (ADASYN)
ada = ADASYN(random_state=369)
X_train_xgb, y_train_xgb = ada.fit_resample(X_train_scaled, y_train)
model_xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=369).fit(X_train_xgb, y_train_xgb)

# --- WEIGHTED VOTING LOOP ---
print(f"{'LGBM Weight':<12} | {'XGB Weight':<12} | {'Accuracy':<10}")
print("-" * 40)

probs_lgbm = model_lgbm.predict_proba(X_test_scaled)
probs_xgb = model_xgb.predict_proba(X_test_scaled)

best_acc = 0
best_w = (0, 0)

# Try weights from 0.0 to 1.0
for w_lgbm in np.arange(0.1, 1.0, 0.1):
    w_xgb = 1.0 - w_lgbm

    # Weighted Average
    avg_probs = (probs_lgbm * w_lgbm) + (probs_xgb * w_xgb)
    y_pred = np.argmax(avg_probs, axis=1)
    acc = accuracy_score(y_test, y_pred)

    print(f"{w_lgbm:.1f}          | {w_xgb:.1f}          | {acc:.4%}")

    if acc > best_acc:
        best_acc = acc
        best_w = (w_lgbm, w_xgb)

print("-" * 40)
print(f"🏆 BEST WEIGHTS: {best_w[0]:.1f} LGBM / {best_w[1]:.1f} XGB -> {best_acc:.4%}")

In [ ]:
# 2. Running the ensemble model with their best weights

# Weighted (lgbm 0.8,xgboost 0.2)
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import ADASYN

# Load & Preprocess Data
df = pd.read_csv('heart.csv')

le = LabelEncoder()
categorical_cols = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']

# Split Data (Seed 369)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=369)

# Scale Data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# TRAIN MODELS ON THEIR SPECIALIZED DATA
# Model A: LightGBM (Using SMOTETomek)
print("Training LightGBM (w/ SMOTETomek)...")
smt = SMOTETomek(random_state=369)
X_train_lgbm, y_train_lgbm = smt.fit_resample(X_train_scaled, y_train)
model_lgbm = LGBMClassifier(random_state=369, verbose=-1)
model_lgbm.fit(X_train_lgbm, y_train_lgbm)

# Model B: XGBoost (Using ADASYN)
print("Training XGBoost (w/ ADASYN)...")
ada = ADASYN(random_state=369)
X_train_xgb, y_train_xgb = ada.fit_resample(X_train_scaled, y_train)
model_xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=369)
model_xgb.fit(X_train_xgb, y_train_xgb)

# APPLY WEIGHTED VOTING (0.8 / 0.2)
# Get probabilities for Class 1 (Heart Disease)
probs_lgbm = model_lgbm.predict_proba(X_test_scaled)
probs_xgb = model_xgb.predict_proba(X_test_scaled)

# Apply Weights
weighted_probs = (probs_lgbm * 0.8) + (probs_xgb * 0.2)

# Convert to final prediction (Class with highest score)
y_pred_ensemble = np.argmax(weighted_probs, axis=1)

# EVALUATE
acc_lgbm = accuracy_score(y_test, model_lgbm.predict(X_test_scaled))
acc_xgb = accuracy_score(y_test, model_xgb.predict(X_test_scaled))
acc_ensemble = accuracy_score(y_test, y_pred_ensemble)

print("\n" + "="*50)
print(f"RESULTS (Seed 369 | Weights: LGBM=0.8, XGB=0.2)")
print("="*50)
print(f"1. LightGBM (Individual):  {acc_lgbm:.4%}")
print(f"2. XGBoost  (Individual):  {acc_xgb:.4%}")
print("-" * 50)
print(f"🏆 WEIGHTED ENSEMBLE:       {acc_ensemble:.4%}")
print("="*50)

if acc_ensemble > max(acc_lgbm, acc_xgb):
    print("✅ SUCCESS: The 60/40 weighting improved the result!")
elif acc_ensemble == max(acc_lgbm, acc_xgb):
    print("⚠️ SAME: The result matched the best individual model.")
else:
    print("❌ DECREASE: This weighting combination performed worse.")